In [1]:
import os
import pandas as pd
import numpy as np

# CONFIGURATION – change this to your actual data folder
DATA_PATH = "/Users/trivenidhamdhere/Documents/MSc Data Science/Data Science Projects and Methods/Final Course Work/research-portfolio-lo25538/Data/eicu_collaborative_research_database_demo-2.0.1/Final Data"

def load_csv(base_name):
    """Load CSV (or gzipped CSV) from DATA_PATH. Returns None if not found."""
    for ext in ['.csv', '.csv.gz']:
        path = os.path.join(DATA_PATH, base_name + ext)
        if os.path.exists(path) and os.path.getsize(path) > 0:
            return pd.read_csv(path, low_memory=False)
    return None


In [2]:
# 1. Load main tables
patient = load_csv('patient')
hospital = load_csv('hospital')
apache = load_csv('apacheApsVar')

if patient is None:
    raise FileNotFoundError("Patient table not found. Check DATA_PATH.")

# Merge patient with hospital for region info
patient_hosp = patient.merge(hospital, on='hospitalid', how='left')


In [3]:
#2. FAIRNESS: demographic distribution (cleaned + robust)

print("FAIRNESS – Demographic distribution (training population)")

"""
    Input:   patient table (columns: ethnicity, gender, age)
    Purpose: Check the training population skewness
    Output:  Ethnicity percentages, gender percentages, age summary.
"""

# Define demographic columns and labels
demographics = {
    'ethnicity': 'Ethnicity',
    'gender': 'Gender',
    'age': 'Age (years)'
}

for col, label in demographics.items():
    if col not in patient.columns:
        print(f"No column '{col}' – skipping {label}")
        continue
    
    data = patient[col].dropna()
    if data.empty:
        print(f"Column '{col}' has no valid data – skipping {label}")
        continue
    
    if col == 'age':
        # Convert to numeric, coercing non-numbers (e.g., "> 89", "Unknown") to NaN
        age_numeric = pd.to_numeric(data, errors='coerce')
        age_clean = age_numeric.dropna()
        if age_clean.empty:
            print(f"Column '{col}' had no convertible numeric ages – skipping")
        else:
            print(f"\n{label}: mean={age_clean.mean():.1f}, median={age_clean.median():.1f}, "
                  f"min={age_clean.min()}, max={age_clean.max()}, n={len(age_clean)}")
    else:
        # Categorical columns
        counts = data.value_counts()
        percentages = counts / len(patient) * 100   # % of total patients (including NaN)
        print(f"\n{label} (% of total patients):")
        for category, cnt in counts.head(10).items():
            print(f"  {str(category):30s} {cnt:6d} ({percentages[category]:5.1f}%)")
        if len(counts) > 10:
            print(f"  ... and {len(counts)-10} more categories")

FAIRNESS – Demographic distribution (training population)

Ethnicity (% of total patients):
  Caucasian                        2010 ( 79.8%)
  African American                  231 (  9.2%)
  Hispanic                          115 (  4.6%)
  Other/Unknown                      83 (  3.3%)
  Asian                              30 (  1.2%)
  Native American                    12 (  0.5%)

Gender (% of total patients):
  Male                             1508 ( 59.8%)
  Female                           1008 ( 40.0%)

Age (years): mean=62.2, median=65.0, min=15.0, max=89.0, n=2418


In [4]:

# 3. FAIRNESS: mortality by region (context variation)
if 'hospitaldischargestatus' in patient_hosp.columns and 'region' in patient_hosp.columns:
    """
        Input:   patient_hosp (merged with hospital, columns: region, hospitaldischargestatus)
        Purpose: Demonstrate meaningful between‑context differences.
        Output:  Table of region counts, hospital counts, mortality rates.
    """
    patient_hosp['died'] = (patient_hosp['hospitaldischargestatus'] == 'Expired').astype(float)
    
    region_stats = patient_hosp.groupby('region').agg(
        n_patients=('patientunitstayid', 'count'),
        n_hospitals=('hospitalid', 'nunique'),
        mortality_rate=('died', 'mean')
    ).round(3)
    
    print("FAIRNESS – Mortality rate by US region (context variation)")
    
    print(region_stats.to_string())
    
    # Highlight small region
    min_region = region_stats['n_patients'].idxmin()
    print(f"\nNote: The smallest region ('{min_region}') has only {region_stats.loc[min_region, 'n_patients']} patients, "
          f"but its mortality rate ({region_stats.loc[min_region, 'mortality_rate']*100:.1f}%) "
          f"differs markedly from others (e.g., Midwest: {region_stats.loc['Midwest', 'mortality_rate']*100:.1f}%). "
          f"This validates meaningful between‑context variation.")

FAIRNESS – Mortality rate by US region (context variation)
           n_patients  n_hospitals  mortality_rate
region                                            
Midwest           807           62           0.072
Northeast         159           13           0.119
South             738           54           0.085
West              606           39           0.083

Note: The smallest region ('Northeast') has only 159 patients, but its mortality rate (11.9%) differs markedly from others (e.g., Midwest: 7.2%). This validates meaningful between‑context variation.


In [5]:
# 4. PRIVACY: small subgroups (hospitals with <10 patients)
print("PRIVACY – Hospital size distribution (re‑identification risk)")


if 'hospitalid' in patient.columns:
    """
        Input:   patient table (hospitalid column)
        Purpose: Identify hospitals with very few patients (<10).
        Output:  Number of tiny hospitals and their patient counts.
    """
    hosp_counts = patient['hospitalid'].value_counts()
    tiny = hosp_counts[hosp_counts < 10]
    print(f"Total hospitals: {len(hosp_counts)}")
    print(f"Hospitals with <10 patients: {len(tiny)} ({len(tiny)/len(hosp_counts)*100:.1f}%)")
    if len(tiny) > 0:
        print("\nSmallest hospitals (patients):")
        print(tiny.sort_values().head(10).to_string())
        print("\n Re‑identification risk: even de‑identified data from such small units may be traceable.")
    else:
        print("No hospital with <10 patients in this demo – but full database may have them.")


PRIVACY – Hospital size distribution (re‑identification risk)
Total hospitals: 186
Hospitals with <10 patients: 0 (0.0%)
No hospital with <10 patients in this demo – but full database may have them.


In [6]:
# 5. GENERAL LIMITATIONS: US‑only and limited diversity

print("GENERAL LIMITATIONS")


if 'region' in patient_hosp.columns:
    """
        Input:   patient_hosp (region column), patient (ethnicity column)
        Purpose: Explicitly note that the data is from US hospitals only,
                and the population is predominantly Caucasian.
        Output:  Statement of observed regions and Caucasian proportion.
    """
    regions = patient_hosp['region'].dropna().unique()
    print(f"All hospitals are in the US (regions observed: {sorted(regions)}).")
    print("Results may not transfer to non‑US healthcare systems.")

# Show that the dataset is predominantly Caucasian
if 'ethnicity' in patient.columns:
    caucasian_pct = (patient['ethnicity'].str.contains('Caucasian', case=False, na=False).sum() / len(patient)) * 100
    print(f"\nCaucasian proportion: {caucasian_pct:.1f}% – limited diversity for fairness testing.")

GENERAL LIMITATIONS
All hospitals are in the US (regions observed: ['Midwest', 'Northeast', 'South', 'West']).
Results may not transfer to non‑US healthcare systems.

Caucasian proportion: 79.8% – limited diversity for fairness testing.


In [7]:
# 6. Additional check: outcome prevalence (prior probability shift)
if 'hospitaldischargestatus' in patient.columns:
    mortality = (patient['hospitaldischargestatus'] == 'Expired').mean()
    print(f"\nOverall in‑hospital mortality: {mortality*100:.1f}%")
    print("If deployment setting has very different mortality, prior probability shift may occur.")


Overall in‑hospital mortality: 8.4%
If deployment setting has very different mortality, prior probability shift may occur.
